# Name That LLM: tuning the LM blend with Optuna

**Goal:** search five interacting knobs of the blend at once (model size, layer, number of
averaged layers, regularization `C`, blend weight), then test whether the tuned blend beats
`lm_blend`, the hand-picked config from the
[sweep notebook](https://www.kaggle.com/code/ryqn00/name-that-llm-lm-layer-sweep).

Part of the learning-in-public MLOps project
[github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm), plan P3b.
Like the sweep, this CPU notebook reads the features that the GPU notebook
[name-that-llm-lm-features](https://www.kaggle.com/code/ryqn00/name-that-llm-lm-features)
extracted.

**How Optuna searches:** its TPE sampler learns from the finished trials which regions of
the search space score well, and samples more there. So 60 trials explore five knobs far
better than a 60-point grid could.

**How we avoid fooling ourselves:** all 60 trials are scored on the **seed-7** folds, like
the sweep. The best of 60 tries on one split is partly luck, so the winner is re-measured
against `lm_blend` on **fresh** folds (seeds 42, 43, 44). It's promoted only if the whole
95% bootstrap interval of the difference is above 0.

In [ ]:
# Pinned to an exact commit, so this run is reproducible even after the branch moves on.
%pip install -q "name-that-llm[tune,track] @ git+https://github.com/Ryqn0/name-that-llm@7987b883f2d442f37ed8d74332e21faf0e67f474"

In [ ]:
import glob
import os
import time

import numpy as np
import optuna
import pandas as pd
import yaml

from name_that_llm.blend import blend
from name_that_llm.config import BlendConfig, load_config
from name_that_llm.data import LABELS, load_train
from name_that_llm.lm import select_layers
from name_that_llm.metrics import prompt_bootstrap_ci, score
from name_that_llm.train import run_cv, run_cv_features
from name_that_llm.tune import best_blend_config, run_study

# Tracking is optional: without the Kaggle Secret, everything still runs.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    TRACK = True
except Exception as exc:
    TRACK = False
    print(f"W&B tracking is OFF ({type(exc).__name__}). Results are still computed.")

train = load_train()

# Row alignment (plan Review Focus 1): the features are plain arrays matched to labels by
# POSITION. Any mismatch would make every number here meaningless, so stop right away.
ids_files = glob.glob("/kaggle/input/**/train_ids.npy", recursive=True)
assert len(ids_files) == 1, f"expected one train_ids.npy, found {ids_files}"
assert (np.load(ids_files[0], allow_pickle=True) == train["id"].to_numpy()).all(), (
    "row order differs!"
)

FEATURES = {}
for path in sorted(glob.glob("/kaggle/input/**/features-*.npy", recursive=True)):
    variation = os.path.basename(path).removeprefix("features-").removesuffix(".npy")
    FEATURES[variation] = np.load(path)
    print(f"{variation:10s} {FEATURES[variation].shape} {FEATURES[variation].dtype}")
assert FEATURES, "no features found: is the lm-features notebook attached?"

y = train["model"].to_numpy()
groups = train["prompt_id"].to_numpy()
labels = np.asarray(LABELS)
BASE = load_config("lm_blend", model_cls=BlendConfig)  # the sweep's pick: the bar to beat
TEXT = load_config(BASE.text_config)


def blend_oof(cfg, text_oof, seed):
    # The blend's out-of-fold probabilities for `cfg` on seed-`seed` folds, plus the LM's CV.
    X = select_layers(FEATURES[cfg.lm_variation], cfg.layer, cfg.layer_window)
    lm = run_cv_features(X, train, C=cfg.C, seed=seed)
    return blend([text_oof, lm.oof], [1 - cfg.weight, cfg.weight]), lm

## 1. The search: 60 trials on the seed-7 folds

Each trial samples a config, cross-validates the LM head on the seed-7 folds, and scores
its blend with `lr`'s out-of-fold probabilities, which are computed once. A trial takes
about a minute. Optuna logs one line per trial.

In [ ]:
start = time.time()
study = run_study(FEATURES, train, n_trials=60, seed=7, text_config=BASE.text_config)
tuned = best_blend_config(study)
print(
    f"{len(study.trials)} trials in {(time.time() - start) / 60:.0f} min, "
    f"best seed-7 CV {study.best_value:.4f}"
)
print(yaml.safe_dump(tuned.model_dump(), sort_keys=False))

# The bar to beat, on the SAME folds. It should reproduce the sweep's number exactly.
text7 = run_cv(TEXT.model_copy(update={"seed": 7}), train)
base7 = score(y, labels[blend_oof(BASE, text7.oof, 7)[0].argmax(axis=1)])
print(f"lm_blend on the same seed-7 folds: {base7:.4f} (the sweep measured 0.4038)")

## 2. How the search went

Each dot is one trial; the line is the best score so far. Then the parameters ranked by how
much of the score's variation they explain. Read the `layer_*` shares with care: each of
those parameters exists only in the trials of its own model size.

In [ ]:
import matplotlib.pyplot as plt

trials = study.trials_dataframe()  # one row per trial: number, value, params_*, user_attrs_*

# Colors: the first two slots of a colorblind-checked palette (validated all-pairs). Text
# stays in ink colors; the reference line is gray and labeled directly.
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
BLUE, ORANGE = "#2a78d6", "#eb6834"

fig, ax = plt.subplots(figsize=(9, 4.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.scatter(trials["number"], trials["value"], s=36, color=BLUE, label="one trial", zorder=3)
ax.plot(
    trials["number"],
    trials["value"].cummax(),
    color=ORANGE,
    linewidth=2,
    label="best so far",
    zorder=2,
)
ax.axhline(base7, color=MUTED, linestyle="--", linewidth=1.5)
ax.annotate(
    "lm_blend (the sweep's pick)",
    (0, base7),
    xytext=(2, 4),
    textcoords="offset points",
    color=MUTED,
    fontsize=9,
)
ax.set_xlabel("trial", color=INK_2)
ax.set_ylabel("blend CV balanced accuracy (seed-7 folds)", color=INK_2)
ax.set_title("Optuna's search over the blend", loc="left", fontweight="bold", color=INK)
ax.grid(color=GRID, linewidth=0.6)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2)
plt.show()

print("parameter importance (share of the score's variation):")
for name, value in optuna.importance.get_param_importances(study).items():
    print(f"  {name:14s} {value:.2f}")

## 3. The honest A/B on fresh folds

The tuned config against `lm_blend`, each blended with the same `lr`, on seeds 42, 43 and
44. The prompt bootstrap on seed 42 decides: promote the tuned config only if the whole 95%
interval of "tuned minus `lm_blend`" is above 0.

In [ ]:
rows, kept = [], {}
for seed in (42, 43, 44):
    text = run_cv(TEXT.model_copy(update={"seed": seed}), train)
    base_mix, base_lm = blend_oof(BASE, text.oof, seed)
    tuned_mix, _ = blend_oof(tuned, text.oof, seed)
    base_pred, tuned_pred = labels[base_mix.argmax(axis=1)], labels[tuned_mix.argmax(axis=1)]
    rows.append(
        {
            "seed": seed,
            "lr": text.score,
            "lm_blend": score(y, base_pred),
            "tuned": score(y, tuned_pred),
        }
    )
    kept[seed] = (base_pred, tuned_pred, text.oof, base_lm.oof)
report = pd.DataFrame(rows).set_index("seed")
report.loc["mean"] = report.mean()
# print(), not just the cell's display: the downloadable run log only keeps printed text.
print(report.round(4).to_string())

diff, lo, hi = prompt_bootstrap_ci(y, kept[42][0], kept[42][1], groups, n_boot=2000, seed=0)
PROMOTE = lo > 0
print(
    f"tuned minus lm_blend (seed 42): {diff:+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}] -> "
    + ("PROMOTE the tuned blend" if PROMOTE else "keep lm_blend: the gain isn't clear")
)

## 4. Why is the best blend weight so small? (idea I-015)

The sweep's best weight on the LM side was only 0.1, although the LM head alone beats `lr`.
Our hypothesis: a weighted mean mixes *probabilities*, and the LM head is far more
confident, so 10% of it already moves the result. If so, its average top probability is
much higher than `lr`'s.

In [ ]:
_, _, text42, lm42 = kept[42]
print(
    f"mean top probability, seed-42 out-of-fold: lr {text42.max(axis=1).mean():.3f}, "
    f"LM head of lm_blend {lm42.max(axis=1).mean():.3f} (uniform: {1 / 18:.3f})"
)

## 5. Outputs

In [ ]:
with open("/kaggle/working/lm_blend_tuned.yaml", "w") as f:
    yaml.safe_dump(tuned.model_dump(), f, sort_keys=False)
trials.to_csv("/kaggle/working/tuning_trials.csv", index=False)
print("outputs:", sorted(os.listdir("/kaggle/working")))

if TRACK:
    import wandb

    run = wandb.init(
        project="name-that-llm",
        job_type="tuning",
        name="lm-blend-tuning",
        config=tuned.model_dump(),
    )
    run.summary.update(
        {
            "best_seed7": study.best_value,
            "lm_blend_seed7": base7,
            "cv_lm_blend_mean": report.loc["mean", "lm_blend"],
            "cv_tuned_mean": report.loc["mean", "tuned"],
            "tuned_minus_lm_blend": diff,
            "ci_low": lo,
            "ci_high": hi,
            "promote": bool(PROMOTE),
        }
    )
    run.finish()

## 6. Findings

From run v1 (2026-10-25, 1 h 48 min on Kaggle's CPU).

**1. Tuning didn't beat the hand-picked blend.** The 60 trials took 81 minutes. The best
one (Qwen 3 1.7B, layer 3 alone, `C = 0.01`, LM weight 0.63) scored 0.4012 on the seed-7
folds, below `lm_blend`'s 0.4038 on the same folds.

**2. Fresh folds agree, so `lm_blend` stays:**

| seed | `lr` | `lm_blend` | tuned |
|---|---|---|---|
| 42 | 0.3212 | 0.4052 | 0.3963 |
| 43 | 0.3300 | 0.4001 | 0.3956 |
| 44 | 0.3252 | 0.4015 | 0.4015 |
| mean | 0.3255 | 0.4023 | 0.3978 |

Tuned minus `lm_blend` on seed 42: −0.009, with a 95% CI of [−0.027, +0.009].

**3. Why the sweep won with a similar budget.** The sweep ran 67 cross-validations, one
knob at a time around the best point, then chose the blend weight on a 21-point grid. That
grid is almost free: once the LM's out-of-fold probabilities exist, trying a weight needs
no refitting. Optuna instead spent whole trials on bad weights. It also spread its 60
trials over three model sizes, each with its own layer knob. And `C` and the weight trade
off: a smaller `C` makes the LM head less confident, so it needs a larger weight (`C` 0.01
with weight 0.63 here, against `C` 0.1 with weight 0.1 for `lm_blend`). TPE samples each
knob independently by default, so it follows such a ridge poorly.

**4. The selection bias is small.** `lm_blend` went from 0.4038 on the folds that chose it
to a mean of 0.4023 on fresh ones: only 0.0015 of its score was luck.

**5. The I-015 hypothesis holds.** On average, `lr` gives its top class a probability of
0.147, and the LM head 0.666 (uniform: 0.056). The head is about 4.5 times more confident,
which is why a weight of 0.1 is enough to move the blend.

**Parameter importance** (share of the score's variation): `layer_1.7b-base` 0.51,
`layer_4b-base` 0.26, `weight` 0.15, `lm_variation` 0.04, `C` 0.03, `layer_window` 0.01.
Choosing the layer is what matters most.

**Next ideas:** choose the weight by grid *inside* each trial, so Optuna only searches the
LM's own knobs (I-018); and blend calibrated or log probabilities, so the weight means what
it says (I-015).